## Imports and Data Loading

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

TARGET = 'addicted_label'
ID_COL = 'id'

print(train.shape, test.shape, sample_submission.shape)

(691369, 14) (296302, 13) (296302, 2)


## Prepare categorical features

In [2]:
categorical_features = ["gender", "stress_level", "academic_work_impact"]

for col in categorical_features:
    train[col] = train[col].fillna("Missing").astype(str)
    test[col] = test[col].fillna("Missing").astype(str)

    # Fit encoding on combined train+test categories so codes match
    combined = pd.concat([train[col], test[col]], axis=0)
    categories = combined.astype("category").cat.categories

    train[col] = pd.Categorical(train[col], categories=categories).codes
    test[col] = pd.Categorical(test[col], categories=categories).codes

print(train[categorical_features].dtypes)
print(test[categorical_features].dtypes)

gender                  int8
stress_level            int8
academic_work_impact    int8
dtype: object
gender                  int8
stress_level            int8
academic_work_impact    int8
dtype: object


## Build X/y

In [3]:
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])

print(X.dtypes)

age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
gender                        int8
stress_level                  int8
academic_work_impact          int8
dtype: object


## Seed-averaged LightGBM ensemble

In [4]:
seeds = [42, 1, 7, 2024, 99]
test_preds_list = []
val_scores = []

for seed in seeds:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    seed_model = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        num_leaves=64,
        random_state=seed,
        n_jobs=-1
    )

    seed_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    val_preds = seed_model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_preds)
    val_scores.append(val_auc)
    print(f"Seed {seed}: Validation AUC = {val_auc:.5f}")

    final_model = lgb.LGBMClassifier(
        n_estimators=seed_model.best_iteration_,
        learning_rate=0.02,
        num_leaves=64,
        random_state=seed,
        n_jobs=-1
    )
    final_model.fit(X, y)
    test_preds_list.append(final_model.predict_proba(X_test)[:, 1])

print(f"\nMean validation AUC across seeds: {np.mean(val_scores):.5f}")

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1957
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[3000]	valid_0's auc: 0.962532	valid_0's binary_logloss: 0.223848
Seed 42: Validation AUC = 0.96253
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006773 seconds.
You can set `force_row_wise=true` to remove the overhe

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007805 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1957
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2998]	valid_0's auc: 0.963508	valid_0's binary_logloss: 0.221138
Seed 1: Validation AUC = 0.96351
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011786 seconds.
You can set `force_row_wise=true` to remove the overhea

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006898 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1957
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2999]	valid_0's auc: 0.962568	valid_0's binary_logloss: 0.22371
Seed 7: Validation AUC = 0.96257
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010440 seconds.
You can set `force_row_wise=true` to remove the overhead

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014697 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1956
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2994]	valid_0's auc: 0.962877	valid_0's binary_logloss: 0.223056
Seed 2024: Validation AUC = 0.96288
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008148 seconds.
You can set `force_row_wise=true` to remove the over

/home/nesrine/miniconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003985 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1957
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2999]	valid_0's auc: 0.963277	valid_0's binary_logloss: 0.221728
Seed 99: Validation AUC = 0.96328
[LightGBM] [Info] Number of positive: 490474, number of negative: 200895
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006175 seconds.
You can set `force_row_wise=true` to remove the overhe

## Average predictions across seeds

In [5]:
final_test_preds = np.mean(test_preds_list, axis=0)

## Submission

In [6]:
submission = sample_submission.copy()
submission[TARGET] = final_test_preds
submission.to_csv("../submissions/submission_ensemble.csv", index=False)
submission.head()

,id,addicted_label
0,691369,0.999041
1,691370,0.966282
2,691371,0.942060
3,691372,0.987730
4,691373,0.997487
